# Drive Model Reliability — Gold View

## Purpose
Provide a model-level view of Backblaze drive observations and reported failures to compare observed reliability across drive models.

**Source:** `backblaze_data.sot_silver.drive_daily_stats`   
**Target:** `backblaze_data.sot_gold_views.drive_model_reliability_vw`  
**Grain:** One row per drive model and source period.  
**Layer Purpose:** Gold — Business-facing analytical view.  

**Calculation:**  
Observed failure percentage =
(Failed drive observations / Total drive observations) * 100  

## Business Metrics

- Total drive observations per model and source period.
- Distinct drives observed per model.
- Number of observations where failure = 1.
- Observed failure percentage per model.

**Note:** This is an observed failure percentage, not an annualized failure rate (AFR). Models may have different observation counts and exposure durations.

In [0]:
%sql
-- Purpose: Aggregate Silver observations by model and period.
CREATE OR REPLACE VIEW backblaze_data.sot_gold_views.drive_model_reliability_vw AS
SELECT
    model,
    _period,
    COUNT(*) AS total_drive_observations,
    COUNT(DISTINCT serial_number) AS distinct_drives_observed,
    SUM(
        CASE
            WHEN failure = 1 THEN 1
            ELSE 0
        END
    ) AS failed_drive_observations,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN failure = 1 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS observed_failure_rate_pct

FROM backblaze_data.sot_silver.drive_daily_stats
GROUP BY
    model,
    _period;

In [0]:
%sql
ALTER VIEW backblaze_data.sot_gold_views.drive_model_reliability_vw
SET TAGS (
    'layer' = 'gold',
    'domain' = 'drive_reliability',
    'subject' = 'model_reliability',
    'refresh_type' = 'on_query'
);

In [0]:
%sql
COMMENT ON VIEW backblaze_data.sot_gold_views.drive_model_reliability_vw IS
'Provides drive model-level observation counts, distinct drive counts, failure counts, and observed failure percentages by source period. The observed failure percentage is not an annualized failure rate.';

In [0]:
%sql
COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_model_reliability_vw.model
IS 'Manufacturer-reported drive model identifier.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_model_reliability_vw._period
IS 'Source dataset period associated with the observations.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_model_reliability_vw.total_drive_observations
IS 'Total number of drive observations recorded for the model and source period.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_model_reliability_vw.distinct_drives_observed
IS 'Number of distinct drive serial numbers observed for the model and source period.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_model_reliability_vw.failed_drive_observations
IS 'Number of drive observations where the failure indicator equals 1.';

COMMENT ON COLUMN backblaze_data.sot_gold_views.drive_model_reliability_vw.observed_failure_rate_pct
IS 'Percentage of drive observations with failure equal to 1 for the model and source period. Not an annualized failure rate.';

In [0]:
%sql
SELECT *
FROM backblaze_data.sot_gold_views.drive_model_reliability_vw
ORDER BY observed_failure_rate_pct DESC
LIMIT 20;

In [0]:
%sql
DESCRIBE TABLE EXTENDED
backblaze_data.sot_gold_views.drive_model_reliability_vw;